# Experiment 4 — Editing and Generation (ROME-style)

**Group 25 · Opening the Black Box · Jesus D. Jimenez Ballestas**

This notebook is the *generative* half of the project. It takes the fact-storing MLP that the
interpretability experiments localize, applies a **rank-one edit** in the style of Meng et al. (2022,
ROME), then has the edited model **generate text** and scores that text on the four standard
factual-editing properties:

| Metric | Question it answers |
|---|---|
| **Efficacy** | Does the edit take? Does the model now generate the *new* object? |
| **Generalization** | Does it hold under paraphrased prompts? |
| **Specificity** | Are unrelated ("neighborhood") facts left untouched? |
| **Fluency** | Does generation stay coherent (not degenerate/repetitive)? |

**Pipeline:** load `pythia-160m` → pick an edit target → causal-trace to find the decisive layer →
optimize a target value `v*` → solve a rank-one update to `W_out` → generate before/after → evaluate.

> **First run:** the setup cell installs `torch`, `transformers`, `transformer_lens` and downloads
> `pythia-160m` (~380 MB). Everything runs on CPU; a GPU just makes it faster.
>
> **Note on scope:** this uses the identity-covariance simplification of ROME (no second-moment
> statistics collected from a corpus). That keeps it fully self-contained and runnable; the
> generalization/specificity metrics will honestly reveal the quality of the simplified edit.

In [ ]:
# --- Setup: install dependencies on first run -------------------------------
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
    except ImportError:
        pip_name = pip_name or import_name
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pip_name.split()])

ensure("torch")
ensure("transformers")
ensure("transformer_lens")
print("Dependencies ready.")

In [ ]:
# --- Imports and configuration ---------------------------------------------
import math
from collections import Counter

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer

MODEL_NAME = "pythia-160m"      # swap to "pythia-410m" for the larger model
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
print("Device:", DEVICE)

## 1. Load the model

In [ ]:
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
model.eval()
print(f"{MODEL_NAME}: n_layers={model.cfg.n_layers}, d_model={model.cfg.d_model}, d_mlp={model.cfg.d_mlp}")

## 2. Edit targets

Each target is a fact `(subject, prompt, true -> new)` plus two evaluation sets:
**paraphrases** (generalization) and **neighborhood** facts that must stay unchanged (specificity).
Keep prompts ending right before the object so the answer is the model's next token.

In [ ]:
EDIT_TARGETS = [
    {
        "subject": "The Eiffel Tower",
        "prompt":  "The Eiffel Tower is located in the city of",
        "true":    "Paris",
        "new":     "Rome",
        "paraphrases": [
            "You can find the Eiffel Tower in the heart of",
            "Tourists who want to see the Eiffel Tower travel to",
        ],
        "neighborhood": [
            ("Big Ben is located in the city of", "London"),
            ("The Brandenburg Gate is located in the city of", "Berlin"),
        ],
    },
    {
        "subject": "Mount Everest",
        "prompt":  "Mount Everest is located in the country of",
        "true":    "Nepal",
        "new":     "Canada",
        "paraphrases": [
            "To climb Mount Everest you would travel to",
            "Mount Everest can be found in",
        ],
        "neighborhood": [
            ("Mount Fuji is located in the country of", "Japan"),
            ("Kilimanjaro is located in the country of", "Tanzania"),
        ],
    },
]

# Work through the first target end-to-end; loop later once it works.
TARGET = EDIT_TARGETS[0]
TARGET

## 3. Helpers

In [ ]:
def first_token(s):
    """Token id of the first BPE token of ' s' (GPT-NeoX prefixes a space)."""
    return model.to_tokens(" " + s.strip(), prepend_bos=False)[0, 0].item()

def next_token_logprobs(prompt):
    """Log-probs over the vocabulary for the token that follows `prompt`."""
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)
    return torch.log_softmax(logits[0, -1], dim=-1)

def top_prediction(prompt, k=5):
    lp = next_token_logprobs(prompt)
    vals, idx = lp.topk(k)
    return [(model.to_string(i.item()), round(v.item(), 3)) for v, i in zip(vals, idx)]

def answer_logprob(prompt, answer):
    return next_token_logprobs(prompt)[first_token(answer)].item()

def subject_span(prompt, subject):
    """(start, end) token positions (inclusive) of `subject` inside `prompt`, BOS-offset."""
    str_toks = model.to_str_tokens(prompt, prepend_bos=True)
    text, spans = "", []
    for i, t in enumerate(str_toks):
        if i == 0:          # BOS
            continue
        start = len(text); text += t; end = len(text)
        spans.append((start, end, i))
    ci = text.find(subject)
    if ci == -1:
        ci = text.find(subject.strip())
    cj = ci + len(subject)
    start_pos = next(i for (s, e, i) in spans if e > ci)
    end_pos   = next(i for (s, e, i) in spans if e >= cj)
    return start_pos, end_pos

def generate(prompt, max_new_tokens=30):
    return model.generate(prompt, max_new_tokens=max_new_tokens,
                          do_sample=False, verbose=False)

def ngram_entropy(text, n=3):
    toks = model.to_str_tokens(text, prepend_bos=False)
    if len(toks) < n:
        return 0.0
    grams = Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))
    total = sum(grams.values())
    return -sum((c / total) * math.log2(c / total) for c in grams.values())

def fluency(text):
    """Mean of bi- and tri-gram entropy; lower => more repetitive/degenerate."""
    return round((ngram_entropy(text, 2) + ngram_entropy(text, 3)) / 2, 3)

print("Sanity check ->", first_token(TARGET["true"]), first_token(TARGET["new"]))
print("Subject span  ->", subject_span(TARGET["prompt"], TARGET["subject"]))

## 4. Baseline (before any edit)

In [ ]:
print("Prompt:", TARGET["prompt"])
print("Top next tokens:", top_prediction(TARGET["prompt"]))
print(f"  logprob(true='{TARGET['true']}') = {answer_logprob(TARGET['prompt'], TARGET['true']):.3f}")
print(f"  logprob(new ='{TARGET['new']}')  = {answer_logprob(TARGET['prompt'], TARGET['new']):.3f}")
print("\nBaseline generation:")
print(" ", generate(TARGET["prompt"]))

## 5. Causal tracing — where does the fact live?

Corrupt the subject tokens with Gaussian noise (~3σ of the embedding matrix), then patch the clean
`resid_post` back in, one (layer, position) at a time, and measure how much of the true-object
probability is restored. The layer that restores the most at the subject's last token is where we
edit.

In [ ]:
def causal_trace(target, noise_scale=3.0):
    prompt, subject, answer = target["prompt"], target["subject"], target["true"]
    tokens = model.to_tokens(prompt)
    n_pos = tokens.shape[1]
    ans_tok = first_token(answer)
    s0, s1 = subject_span(prompt, subject)
    subj_pos = list(range(s0, s1 + 1))

    # clean run
    _, clean_cache = model.run_with_cache(tokens)
    clean_score = torch.log_softmax(model(tokens)[0, -1], dim=-1)[ans_tok].item()

    # fixed corruption noise
    emb_std = model.W_E.std().item()
    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    noise = torch.randn(len(subj_pos), model.cfg.d_model, generator=g, device=DEVICE) * emb_std * noise_scale

    def corrupt_hook(value, hook):
        for j, p in enumerate(subj_pos):
            value[0, p] = value[0, p] + noise[j]
        return value

    corr_logits = model.run_with_hooks(tokens, fwd_hooks=[("hook_embed", corrupt_hook)])
    corr_score = torch.log_softmax(corr_logits[0, -1], dim=-1)[ans_tok].item()

    grid = np.zeros((model.cfg.n_layers, n_pos))
    for l in range(model.cfg.n_layers):
        clean_resid = clean_cache[f"blocks.{l}.hook_resid_post"]
        for p in range(n_pos):
            def patch_hook(value, hook, l=l, p=p, clean_resid=clean_resid):
                value[0, p] = clean_resid[0, p]
                return value
            logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[("hook_embed", corrupt_hook),
                           (f"blocks.{l}.hook_resid_post", patch_hook)])
            grid[l, p] = torch.log_softmax(logits[0, -1], dim=-1)[ans_tok].item()

    recovery = (grid - corr_score) / (clean_score - corr_score + 1e-9)
    return recovery, subj_pos, s1, model.to_str_tokens(prompt, prepend_bos=True)

recovery, subj_pos, subj_last, str_toks = causal_trace(TARGET)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(recovery, aspect="auto", cmap="magma", vmin=0, vmax=1)
ax.set_xticks(range(len(str_toks)))
ax.set_xticklabels([t.strip() or "·" for t in str_toks], rotation=60, ha="right", fontsize=8)
ax.set_yticks(range(model.cfg.n_layers))
ax.set_ylabel("layer"); ax.set_title("Causal-trace recovery of true-object probability")
plt.colorbar(im, label="fraction restored"); plt.tight_layout(); plt.show()

EDIT_LAYER = int(np.argmax(recovery[:, subj_last]))
print(f"Editing at layer {EDIT_LAYER}, subject-last token position {subj_last} "
      f"('{str_toks[subj_last].strip()}')")

## 6. Optimize the target value $v^*$

Add a trainable vector to the layer's `hook_mlp_out` at the subject's last token and push it, by
gradient descent, until the model predicts the **new** object. The optimized MLP output at that
position becomes $v^*$.

In [ ]:
def optimize_v_star(target, layer, pos, n_steps=25, lr=0.5, kl_weight=0.0625):
    prompt = target["prompt"]
    tokens = model.to_tokens(prompt)
    tgt_tok = first_token(target["new"])

    _, clean_cache = model.run_with_cache(tokens)
    v_orig = clean_cache[f"blocks.{layer}.hook_mlp_out"][0, pos].detach()
    k_vec  = clean_cache[f"blocks.{layer}.mlp.hook_post"][0, pos].detach()

    delta = torch.zeros(model.cfg.d_model, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)

    def add_delta(value, hook):
        value[0, pos] = value[0, pos] + delta
        return value

    for step in range(n_steps):
        opt.zero_grad()
        logits = model.run_with_hooks(
            tokens, fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", add_delta)])
        logp = torch.log_softmax(logits[0, -1], dim=-1)
        loss = -logp[tgt_tok] + kl_weight * delta.norm() ** 2
        loss.backward(); opt.step()
        if step % 5 == 0 or step == n_steps - 1:
            print(f"  step {step:2d}: loss={loss.item():.3f}  logp(new)={logp[tgt_tok].item():.3f}")

    v_star = (v_orig + delta.detach())
    return v_star, v_orig, k_vec

v_star, v_orig, k_vec = optimize_v_star(TARGET, EDIT_LAYER, subj_last)

## 7. Solve and apply the rank-one edit

We want the MLP down-projection to map the key $k$ to $v^*$. With
$W_{\text{out}}' = W_{\text{out}} + \dfrac{k}{k\cdot k}\,(v^* - v_{\text{orig}})^\top$,
the layer outputs $v^*$ for input $k$ while changing every other direction minimally. Keep the
original weights so the edit can be undone.

In [ ]:
_ORIGINAL_W_OUT = {}

def apply_rank_one_edit(layer, k_vec, v_star, v_orig):
    W = model.blocks[layer].mlp.W_out          # [d_mlp, d_model]
    if layer not in _ORIGINAL_W_OUT:
        _ORIGINAL_W_OUT[layer] = W.detach().clone()
    r = (v_star - v_orig)                       # residual to inject (== optimized delta)
    k_unit = k_vec / (k_vec @ k_vec + 1e-9)
    with torch.no_grad():
        W += torch.outer(k_unit, r)

def restore_weights():
    with torch.no_grad():
        for layer, W0 in _ORIGINAL_W_OUT.items():
            model.blocks[layer].mlp.W_out.copy_(W0)
    _ORIGINAL_W_OUT.clear()

apply_rank_one_edit(EDIT_LAYER, k_vec, v_star, v_orig)
print("Edit applied.")
print("Top next tokens after edit:", top_prediction(TARGET["prompt"]))
print("Generation after edit:\n ", generate(TARGET["prompt"]))

## 8. Evaluate the generated outputs

Efficacy, generalization, and specificity are read off the model's generations and next-token
predictions; fluency is the n-gram entropy of the generated text. Run once **before** and once
**after** the edit (weights are restored between runs) so every number is a genuine before/after.

In [ ]:
def evaluate(target, layer):
    prompt, true, new = target["prompt"], target["true"], target["new"]

    def snapshot():
        rows = []
        # efficacy (the edited prompt itself)
        gen = generate(prompt)
        rows.append(dict(kind="efficacy", prompt=prompt,
                         pred=top_prediction(prompt, 1)[0][0].strip(),
                         says_new=new.lower() in gen.lower(),
                         says_true=true.lower() in gen.lower(),
                         fluency=fluency(gen), generation=gen))
        # generalization (paraphrases -> should also say NEW)
        for pp in target["paraphrases"]:
            gen = generate(pp)
            rows.append(dict(kind="generalization", prompt=pp,
                             pred=top_prediction(pp, 1)[0][0].strip(),
                             says_new=new.lower() in gen.lower(),
                             says_true=true.lower() in gen.lower(),
                             fluency=fluency(gen), generation=gen))
        # specificity (neighborhood -> should KEEP its own answer)
        for np_prompt, np_ans in target["neighborhood"]:
            gen = generate(np_prompt)
            rows.append(dict(kind="specificity", prompt=np_prompt,
                             pred=top_prediction(np_prompt, 1)[0][0].strip(),
                             keeps_answer=np_ans.lower() in gen.lower(),
                             fluency=fluency(gen), generation=gen))
        return pd.DataFrame(rows)

    restore_weights()                      # ensure clean baseline
    before = snapshot(); before["phase"] = "before"

    # rebuild + apply the edit
    pos = subject_span(prompt, target["subject"])[1]
    vs, vo, kv = optimize_v_star(target, layer, pos)
    apply_rank_one_edit(layer, kv, vs, vo)
    after = snapshot(); after["phase"] = "after"
    restore_weights()

    return pd.concat([before, after], ignore_index=True)

results = evaluate(TARGET, EDIT_LAYER)
pd.set_option("display.max_colwidth", 60)
results[["phase", "kind", "prompt", "pred", "says_new", "says_true", "keeps_answer", "fluency"]]

## 9. Summary scores and figure

In [ ]:
def scores(df):
    eff = df[(df.phase == "after") & (df.kind == "efficacy")]["says_new"].mean()
    gen = df[(df.phase == "after") & (df.kind == "generalization")]["says_new"].mean()
    spec = df[(df.phase == "after") & (df.kind == "specificity")]["keeps_answer"].mean()
    flu = df[df.phase == "after"]["fluency"].mean()
    flu_before = df[df.phase == "before"]["fluency"].mean()
    return dict(efficacy=eff, generalization=gen, specificity=spec,
                fluency_after=round(flu, 3), fluency_before=round(flu_before, 3))

s = scores(results)
print(s)

fig, ax = plt.subplots(figsize=(6, 4))
names = ["efficacy", "generalization", "specificity"]
ax.bar(names, [s[n] for n in names], color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylim(0, 1); ax.set_ylabel("rate")
ax.set_title(f"Edit quality — {MODEL_NAME}, layer {EDIT_LAYER}\n"
             f"fluency {s['fluency_before']} -> {s['fluency_after']}")
for i, n in enumerate(names):
    ax.text(i, s[n] + 0.02, f"{s[n]:.2f}", ha="center")
plt.tight_layout(); plt.show()

## 10. Next steps

- **Loop over all `EDIT_TARGETS`** and aggregate efficacy / generalization / specificity / fluency
  into one table — that table is the deliverable for RQ3.
- **Compare layers:** sweep `EDIT_LAYER` across the causal-trace peak ±2 and show the quality curve.
- **GPT-2 replication (Experiment 5):** set `MODEL_NAME = "gpt2"` and rerun to show the pipeline
  is not Pythia-specific.
- **Toward full ROME:** replace the identity-covariance step with second-moment statistics
  estimated from Pile samples (Meng et al., §3) and compare specificity — the simplified edit here
  is the honest baseline that improvement is measured against.
- **Fluency guardrail:** flag any generation whose entropy drops sharply after the edit as a
  degenerate/over-edited case.